# 2. Merge Single-Cell Profiles

## Purpose
This notebook reads the per-compartment DuckDB produced by notebook 1 for a single
well-FOV and merges the Nuclei, Cell, and Cytoplasm tables into a single-cell (SC)
parquet profile. Organoid and Nucleocentric profiles are passed through and saved as
separate parquets.

This is **step 2 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
- `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/{well_fov}.duckdb`
  - Five compartment tables: `Organoid`, `Nuclei`, `Cell`, `Cytoplasm`, `Nucleocentric`
  - Produced by notebook 1 (`1.merge_feature_parquets.ipynb`)

## Outputs
Three parquet files written to `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:

| File | Content | Rows |
|---|---|---|
| `sc_profiles_{well_fov}.parquet` | Merged Nuclei + Cell + Cytoplasm features | One row per object present in all three compartments |
| `organoid_profiles_{well_fov}.parquet` | Organoid features passed through | One row per segmented organoid |
| `nucleocentric_profiles_{well_fov}.parquet` | Nucleocentric features passed through | One row per nucleus-centered volume |

## Notes
- Only objects present in **all three** of Nuclei, Cell, and Cytoplasm are retained in the SC profile.
  Objects segmented in only some compartments are dropped.
- Object IDs are reassigned to a sequential `1..N` range at the end of this notebook.
  The original segmentation mask IDs are not preserved.

In [1]:
import os
import pathlib

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C10-2"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Load all five compartment tables from the DuckDB produced by notebook 1.
with duckdb.connect(input_sqlite_file) as con:
    tables = con.execute("SHOW TABLES").fetchdf()
    print(tables)
    nuclei_table = con.sql("SELECT * FROM Nuclei").df()
    cells_table = con.sql("SELECT * FROM Cell").df()
    cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
    organoid_table = con.sql("SELECT * FROM Organoid").df()
    nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()

            name
0           Cell
1      Cytoplasm
2         Nuclei
3  Nucleocentric
4       Organoid


In [5]:
# Retain only objects that were successfully segmented in all three compartments.
# A nucleus without a matched cell/cytoplasm (or vice versa) is not a valid
# single-cell profile and is dropped here.
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())

# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)

# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# Merge the three compartment tables into a single-cell dataframe.
# Because object_ids were already filtered to the intersection in the cell above,
# this LEFT JOIN is effectively an INNER JOIN — no NaN-filled rows will result.
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

In [7]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (3, 3961)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C10-2,1210841.0,838.615864,429.637973,9.794105,1825596.0,684,998,277,...,1152.351969,1105.363718,1147.235766,1103.937977,1143.647327,1149.664104,1149.646407,1102.551268,1144.905563,1151.809001
1,2,C10-2,749790.0,304.417777,879.567732,9.106898,1071408.0,207,415,734,...,303.886884,303.366418,304.176987,302.987602,303.820368,303.924470,303.913711,303.237717,303.886440,303.678211
2,3,C10-2,15506.0,987.558365,1484.026699,1.500000,19980.0,934,1045,1437,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (11, 11883)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_DNA_Texture_Variance-3-03-256,Cytoplasm_DNA_Texture_Variance-3-04-256,Cytoplasm_DNA_Texture_Variance-3-05-256,Cytoplasm_DNA_Texture_Variance-3-06-256,Cytoplasm_DNA_Texture_Variance-3-07-256,Cytoplasm_DNA_Texture_Variance-3-08-256,Cytoplasm_DNA_Texture_Variance-3-09-256,Cytoplasm_DNA_Texture_Variance-3-10-256,Cytoplasm_DNA_Texture_Variance-3-11-256,Cytoplasm_DNA_Texture_Variance-3-12-256
0,257,C10-2,53529.0,366.587308,869.943171,4.558576,90720.0,330,402,808,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,514,C10-2,43943.0,314.491660,815.983797,5.786837,75174.0,284,351,770,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1028,C10-2,42928.0,241.827199,863.996436,5.076756,67338.0,214,272,800,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1285,C10-2,25653.0,276.793552,930.398979,2.837212,40180.0,239,321,882,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1542,C10-2,9672.0,577.095327,934.242349,1.500000,15106.0,532,615,886,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (11, 3074)


,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature90,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99
0,257,C10-2,-0.003565,-0.221765,0.128936,0.045298,-0.044059,0.301420,0.124469,-0.107604,...,-1.669074,5.728443,-9.407360,-2.038582,1.370812,-5.477564,-0.336243,1.502100,1.546016,0.210300
1,514,C10-2,-0.448317,-0.227693,0.127213,-0.005069,0.028333,0.112086,0.028173,-0.080888,...,-0.845948,1.572523,-6.513134,-3.327943,5.796319,-3.204541,-0.909036,3.659749,-3.007935,1.133443
2,1028,C10-2,-0.372906,-0.141308,0.181552,-0.005005,0.046912,0.081046,0.008974,-0.084817,...,-2.569808,2.079188,-8.914544,-0.971781,3.080546,-6.735629,-1.390779,2.282022,1.280657,-0.801634
3,1285,C10-2,-0.173261,-0.330922,0.239251,0.008114,-0.084119,0.230205,0.123472,-0.168975,...,-2.485241,3.528395,-4.063998,-3.547382,2.289299,-2.786486,0.148371,3.511132,-2.909451,0.473595
4,1542,C10-2,-0.477709,-0.282062,0.235665,-0.032700,-0.003505,0.119395,0.032428,-0.049148,...,-0.873972,-2.390701,-4.856587,-3.907551,6.513190,3.495226,-0.992177,3.060692,-1.848124,-4.039866


In [10]:
[x for x in nucleocentric_table.columns if "_x" in x or "_y" in x]

[]